0. Intialisation And Import

In [2]:

%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import os
os.chdir(r'C:\Users\benjo\Documents\Projects\ecg-risk-stratification')
print(os.getcwd())

import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


C:\Users\benjo\Documents\Projects\ecg-risk-stratification


In [3]:
binary_df = pd.read_pickle('data/ptbxl_binary_metadata.pkl')
binary_df.head(10)


,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr,diagnostic_superclass,label
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,...,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr,[NORM],0
2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,...,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr,[NORM],0
3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,sinusrhythmus normales ekg,...,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr,[NORM],0
4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,sinusrhythmus normales ekg,...,NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr,[NORM],0
5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,sinusrhythmus normales ekg,...,NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr,[NORM],0
6,19005.0,18.0,1,NaN,58.0,2.0,0.0,CS-12 E,1984-11-28 13:32:13,sinusrhythmus normales ekg,...,NaN,NaN,NaN,NaN,NaN,4,records100/00000/00006_lr,records500/00000/00006_hr,[NORM],0
7,16193.0,54.0,0,NaN,83.0,2.0,0.0,CS-12 E,1984-11-28 13:32:22,"sinusrhythmus linkstyp t abnormal, wahrscheinl...",...,NaN,NaN,NaN,NaN,NaN,7,records100/00000/00007_lr,records500/00000/00007_hr,[NORM],0
9,18792.0,55.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-12-08 09:44:43,sinusrhythmus normales ekg,...,", I-AVR,",NaN,NaN,NaN,NaN,10,records100/00000/00009_lr,records500/00000/00009_hr,[NORM],0
10,9456.0,22.0,1,NaN,56.0,2.0,0.0,CS-12 E,1984-12-12 14:12:46,sinusrhythmus normales ekg,...,NaN,NaN,NaN,NaN,NaN,9,records100/00000/00010_lr,records500/00000/00010_hr,[NORM],0


1. Splitting of the Data into training validation and testing folds, this is provided by the dataset itself

In [4]:
train_df = binary_df[binary_df["strat_fold"].isin(range(1, 9))].copy()
val_df = binary_df[binary_df["strat_fold"] == 9].copy()
test_df = binary_df[binary_df["strat_fold"] == 10].copy()
#Testing for Leakage

train_patients = set(train_df["patient_id"])
val_patients = set(val_df["patient_id"])
test_patients = set(test_df["patient_id"])


print(len(train_patients & val_patients))
print(len(train_patients & test_patients))
print(len(val_patients & test_patients))#Note: in sets the & is the intersection and length shiows how much overlap

0
0
0


In [5]:
print("Train labels:")
print(train_df["label"].value_counts(normalize=True))
print("Val labels:")
print(val_df["label"].value_counts(normalize=True))
print("Test labels:")
print(test_df["label"].value_counts(normalize=True))

Train labels:
label
0    0.633739
1    0.366261
Name: proportion, dtype: float64
Val labels:
label
0    0.633842
1    0.366158
Name: proportion, dtype: float64
Test labels:
label
0    0.636427
1    0.363573
Name: proportion, dtype: float64


2. Pre-Processing

In [6]:
from src.preprocessing import load_and_preprocess
from src.data_extraction import load_ecg

ecg_id = binary_df.index[0]

signal, metadata, row = load_and_preprocess(
    ecg_id,
    binary_df,
    "data/",
    sampling_rate=500
)

print("ECG ID:", ecg_id)
print("Signal shape:", signal.shape)
print("Sampling frequency:", metadata["fs"])
print("Lead names:", metadata["sig_name"])
print("Label:", row["label"])

print("Mean per lead:")
print(signal.mean(axis=1))

print("Std per lead:")
print(signal.std(axis=1))

ECG ID: 1
Signal shape: (12, 5000)
Sampling frequency: 500
Lead names: ['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
Label: 0
Mean per lead:
[ 1.3732910e-08 -3.0517577e-09  6.1035155e-09  9.9182129e-09
 -2.2888185e-09  8.3923339e-09  3.0517577e-09  5.3405760e-09
  1.5258789e-09 -2.2888185e-09  4.5776369e-09  1.6403199e-08]
Std per lead:
[1.         1.         0.9999997  0.9999999  0.99999976 0.9999997
 0.99999994 0.9999998  0.99999994 0.9999999  0.9999999  0.99999976]


3. Creating the Correct Dataset

In [11]:
from src.dataset import ECGdataset

train_dataset = ECGdataset(
    train_df,
    "data/",
    samp_rate=500,
    use_filt=True,
)

test_dataset = ECGdataset(
    test_df,
    "data/",
    samp_rate=500,
    use_filt=True,
)

val_dataset = ECGdataset(
    val_df,
    "data/",
    samp_rate=500,
    use_filt=True,
)



4. Dataloader creation 

In [12]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)
